In [10]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna

# load data (if not already loaded in this notebook)
train_fe = pd.read_parquet("../data/processed/train_fe.parquet")
val_fe = pd.read_parquet("../data/processed/val_fe.parquet")

target = "sales"
exclude_cols = ["sales", "date", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
features = [c for c in train_fe.columns if c not in exclude_cols]

# sample for tuning (same as grid search)
train_sample = train_fe.sample(n=300_000, random_state=42)
X_train_s = train_sample[features]
y_train_s = train_sample[target]

X_val = val_fe[features]
y_val = val_fe[target]

print(X_train_s.shape, X_val.shape)

(300000, 14) (85372, 14)


In [12]:

def objective(trial):

    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "random_state": 42,
        "n_jobs": -1
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train_s, y_train_s)

    preds = model.predict(X_val)
    rmse = np.sqrt(np.mean((y_val - preds) ** 2))

    return rmse

In [13]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best RMSE:", study.best_value)

[I 2026-02-28 21:09:56,731] A new study created in memory with name: no-name-a5b92003-f7f5-4b5b-bc0f-899e42fbbec8
[I 2026-02-28 21:10:04,762] Trial 0 finished with value: 2.0816375902014213 and parameters: {'learning_rate': 0.02765492982523908, 'num_leaves': 198, 'max_depth': 14, 'min_child_samples': 100, 'subsample': 0.9176114340087915, 'colsample_bytree': 0.7071632630685905, 'n_estimators': 595}. Best is trial 0 with value: 2.0816375902014213.
[I 2026-02-28 21:10:06,961] Trial 1 finished with value: 2.0794125210348624 and parameters: {'learning_rate': 0.14675679296833802, 'num_leaves': 137, 'max_depth': 10, 'min_child_samples': 39, 'subsample': 0.8742577799295024, 'colsample_bytree': 0.6594476708825504, 'n_estimators': 226}. Best is trial 1 with value: 2.0794125210348624.
[I 2026-02-28 21:10:12,645] Trial 2 finished with value: 2.102295337881519 and parameters: {'learning_rate': 0.11889722280216526, 'num_leaves': 74, 'max_depth': 12, 'min_child_samples': 71, 'subsample': 0.9265839693

Best Params: {'learning_rate': 0.03211941683103994, 'num_leaves': 63, 'max_depth': 12, 'min_child_samples': 15, 'subsample': 0.7535656471388966, 'colsample_bytree': 0.9663119772895014, 'n_estimators': 264}
Best RMSE: 2.0533826977491785


In [14]:
import os, json
os.makedirs("../models", exist_ok=True)

with open("../models/optuna_best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)

print("Saved ../models/optuna_best_params.json")

Saved ../models/optuna_best_params.json


In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.mean(np.where(denom == 0, 0, np.abs(y_true - y_pred) / denom)) * 100

    return rmse, mae, smape

In [18]:
final_model = lgb.LGBMRegressor(
    **study.best_params,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# FULL training data (not sample)
X_train_full = train_fe[features]
y_train_full = train_fe[target]

final_model.fit(X_train_full, y_train_full)

final_preds = final_model.predict(X_val)

rmse, mae, smape = evaluate(y_val, final_preds)

print("Final Bayesian Model (Full Train)")
print("RMSE:", rmse)
print("MAE:", mae)
print("SMAPE:", smape)

Final Bayesian Model (Full Train)
RMSE: 2.0351186368461063
MAE: 1.0442711251123065
SMAPE: 134.65916370350567


In [19]:
print("=== FINAL MODEL COMPARISON ===")
print("Linear Regression RMSE: 2.1219")
print("Default LightGBM RMSE: 2.0378")
print("Grid Search RMSE: 2.0599")
print("Bayesian (Full Train) RMSE: 2.0351")

=== FINAL MODEL COMPARISON ===
Linear Regression RMSE: 2.1219
Default LightGBM RMSE: 2.0378
Grid Search RMSE: 2.0599
Bayesian (Full Train) RMSE: 2.0351


In [ ]:
=== The Bayesian-optimised LightGBM model achieved the best validation performance ===
=== (RMSE = 2.0351), slightly outperforming the default configuration. Grid search ===
=== did not improve generalisation, demonstrating that exhaustive search does not guarantee ===
=== better performance under time-aware validation.===
